&nbsp;

&nbsp;

# HN-score transcript up5p selection

- [./02_1_HNscore_transcript_fantabio.ipynb](./02_1_HNscore_transcript_fantabio.ipynb) で、全ての HN-score を算出した transcript と CRE の対応を作成したので、これを元に、up5p に対応する transcript-cre を選抜する

In [1]:
import os
import datetime
import pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import polars as pl
import matplotlib.pyplot as plt
from upsetplot import UpSet, from_contents
from IPython.display import display


os.environ["POLARS_MAX_THREADS"] = "16"

In [2]:
VERSION = datetime.datetime.now().strftime("%y%m")
Execute_date = datetime.datetime.now().strftime("%y%m%d")
print(f"Execute Date: {Execute_date}")

Execute Date: 261001


&nbsp;

&nbsp;

&nbsp;

## 1． All HN-score calculated Transcript - CRE pairs

- 公共遺伝子発現データ (bulk RNA-seq) のメタ解析 によって得られた transcriptsのデータを使用する
- このデータには、CRE と対応づいていない transcript も含まれていることに注意
- [./02_1_HNscore_transcript_fantabio.ipynb](./02_1_HNscore_transcript_fantabio.ipynb) によって作成した対応表を使用する

In [3]:
HNscore_transcript_all_fantabio = pl.read_parquet(
    f"../../out/fantabio/parquet_file/HNscore_transcript_all_fantabio_v261001.parquet"
)

pl.Config.set_tbl_rows(4)
display(HNscore_transcript_all_fantabio)

transcript_id,UP2,DOWN2,UNCHANGED2,HN2,UP2_stranded,DOWN2_stranded,UNCHANGED2_stranded,HN2_stranded,gene_symbol,HGNC_ID,transcript_type,gene_id,tags,strand,number_of_tags,RNACentral_ID,corrrespondence,polyA_features,gene_source,annotation_remarks,annotation_remark_count,cre_id,cre_chrom,cre_chrom_start,cre_chrom_end,cre_name,tss_nearest_transcript_ids,tss_nearest_transcript_distance,transcript_distance,screen_ccres,chip_atlas_antigens,silencerdbs,fantom5_cage_peaks,fantom5_enhancers,directionality,fantabio_cre_class,itemRgb,itemRgb_name
str,i64,i64,i64,i64,i64,i64,i64,i64,str,str,str,str,list[str],str,u32,str,str,list[str],str,list[str],u32,str,str,i64,str,str,list[str],str,i64,list[struct[2]],list[struct[3]],list[struct[5]],list[struct[2]],list[struct[1]],f64,str,str,str
"""ENST00000309758.6""",352,17,37,335,232,4,15,228,"""HSPA6""","""HGNC:5239""","""protein_coding""","""ENSG00000173110.8""","[""basic"", ""Ensembl_canonical"", … ""CCDS""]","""+""",6,null,null,null,"""ensembl_havana_gene_homo_sapie…","[""MANE_select""]",1,"""FCHS_28105""","""chr1""",161523745,"""161524487""","""cp1@HSPA6""","[""GQ891451""]","""0""",53,"[{""EH38E2843853"",""pELS""}, {""EH38E2843855"",""PLS""}, {""EH38E3981533"",""pELS""}]","[{""STAG2"",4103,[{""SRX9975614"",4103}, {""SRX9975613"",3500}, {""SRX26159208"",562}]}, {""HSF1"",4038,[{""SRX16646095"",4038}, {""SRX16646111"",3572}, … {""SRX13845684"",554}]}, … {""GABPA"",500,[{""SRX2635792"",500}]}]","[{""HS_00013400"",""H1 BMP4 Derived Mesendoderm Cultured Cells"",""Mesendoderm cell"",""Others"",""SVM""}, {""HS_00013401"",""H1 Cells"",""Embryonic stem cell"",""Stem cell"",""SVM""}, … {""HS_00013403"",""Foreskin Fibroblast Primary Cells skin02"",""Prepuce"",""Skin"",""SVM""}]",null,null,0.982111,"""PLA""","""204,0,0""","""red"""
"""ENST00000309758.6""",352,17,37,335,232,4,15,228,"""HSPA6""","""HGNC:5239""","""protein_coding""","""ENSG00000173110.8""","[""basic"", ""Ensembl_canonical"", … ""CCDS""]","""+""",6,null,null,null,"""ensembl_havana_gene_homo_sapie…","[""MANE_select""]",1,"""FCHS_28106""","""chr1""",161524161,"""161524546""","""cp4@HSPA6""","[""ENST00000309758.6"", ""NM_002155.5"", … ""AK295212""]","""0""",0,"[{""EH38E2843855"",""PLS""}]","[{""HSF1"",4038,[{""SRX16646095"",4038}, {""SRX16646111"",3572}, … {""SRX3322007"",560}]}, {""CTCF"",3651,[{""SRX10975999"",3651}, {""SRX10976007"",2920}, … {""SRX13551280"",511}]}, … {""MED12"",511,[{""SRX10852993"",511}]}]","[{""HS_00013400"",""H1 BMP4 Derived Mesendoderm Cultured Cells"",""Mesendoderm cell"",""Others"",""SVM""}, {""HS_00013401"",""H1 Cells"",""Embryonic stem cell"",""Stem cell"",""SVM""}, … {""HS_00013403"",""Foreskin Fibroblast Primary Cells skin02"",""Prepuce"",""Skin"",""SVM""}]","[{""hg19::chr1:161494328..161494332,+;hg_6264.1"",""p1@HSPA6""}]",null,0.943234,"""PLA""","""204,0,0""","""red"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ENST00000549545.1""",11,265,130,-254,8,174,69,-166,"""COQ10A""","""HGNC:26515""","""retained_intron""","""ENSG00000135469.15""",[],"""+""",0,null,null,null,"""ensembl_havana_gene_homo_sapie…",null,0,"""FCHS_96252""","""chr12""",56268140,"""56268446""","""cp6@COQ10A""","[""ENST00000549545.1""]","""-23""",-23,"[{""EH38E4064794"",""pELS""}]",null,"[{""HS_00173371"",""H1 BMP4 Derived Mesendoderm Cultured Cells"",""Mesendoderm cell"",""Others"",""SVM""}, {""HS_02053667"",""Thyroid Gland"",""Thyroid gland"",""Others"",""gkmSVM""}]",null,null,-0.368421,"""ELA""","""255,202,74""","""yellow"""
"""ENST00000560179.2""",11,272,123,-261,5,181,65,-176,"""ETFA""","""HGNC:3481""","""retained_intron""","""ENSG00000140374.18""","[""RNA_Seq_supported_partial""]","""-""",1,null,null,null,"""ensembl_havana_gene_homo_sapie…","[""RNA-Seq supported partial""]",1,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null


&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 2. up5p transcripts (all datasets)

- related notebook: [../01_HNscore/01_2_HNscore_transcript_gencode.ipynb](../01_HNscore/01_2_HNscore_transcript_gencode.ipynb)
- 収集した全ての bulk RNA-seq から計算した HN-score のデータから選抜したデータを使用する

In [4]:

up5p_transcripts_list = pl.read_csv(
    "../../Data/HNscore/HNscore_transcript/HNscore_gencode_2610/HN2_top_5_percent_transcripts.tsv",
    separator="\t",
    n_threads=16
)


pl.Config.set_tbl_rows(4)
print(up5p_transcripts_list.select("transcript_id").n_unique())
display(up5p_transcripts_list)

26354


transcript_id,UP2,DOWN2,UNCHANGED2,HN2,UPcount+1 / DOWNcount+1 2,log2(UPcount+1 / DOWNcount+1) 2
str,i64,i64,i64,i64,f64,f64
"""ENST00000309758.6""",352,17,37,335,19.611111,4.293599
"""ENST00000679223.2""",338,8,60,330,37.666667,5.235216
…,…,…,…,…,…,…
"""ENST00000797768.1""",92,48,266,44,1.897959,0.924449
"""ENST00000781552.1""",115,71,220,44,1.611111,0.688056


&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 3. `JOIN` Dataframe

In [5]:
up5p_fantabio = up5p_transcripts_list.select("transcript_id").unique().join(
    HNscore_transcript_all_fantabio,
    on="transcript_id",
    how="inner",
    coalesce=True
).sort(
    "HN2",
    descending=True
).filter(
    pl.col("cre_id").is_not_null()
)

pl.Config.set_tbl_rows(2)
print(f"up5p_transcripts_all ensembl transcript count: {up5p_transcripts_list.select('transcript_id').n_unique()}")
print(f"up5p_fantabio ensembl transcript count (has fantabio CRE annotation): {up5p_fantabio.select('transcript_id').n_unique()}")
print(f"up5p_fantabio cre count: {up5p_fantabio.select('cre_id').n_unique()}")
print(f"up5p_fantabio transcript-CRE pairs count: {up5p_fantabio.select(['transcript_id', 'cre_id']).n_unique()}")
display(up5p_fantabio)


up5p_transcripts_all ensembl transcript count: 26354
up5p_fantabio ensembl transcript count (has fantabio CRE annotation): 21696
up5p_fantabio cre count: 33961
up5p_fantabio transcript-CRE pairs count: 63099


transcript_id,UP2,DOWN2,UNCHANGED2,HN2,UP2_stranded,DOWN2_stranded,UNCHANGED2_stranded,HN2_stranded,gene_symbol,HGNC_ID,transcript_type,gene_id,tags,strand,number_of_tags,RNACentral_ID,corrrespondence,polyA_features,gene_source,annotation_remarks,annotation_remark_count,cre_id,cre_chrom,cre_chrom_start,cre_chrom_end,cre_name,tss_nearest_transcript_ids,tss_nearest_transcript_distance,transcript_distance,screen_ccres,chip_atlas_antigens,silencerdbs,fantom5_cage_peaks,fantom5_enhancers,directionality,fantabio_cre_class,itemRgb,itemRgb_name
str,i64,i64,i64,i64,i64,i64,i64,i64,str,str,str,str,list[str],str,u32,str,str,list[str],str,list[str],u32,str,str,i64,str,str,list[str],str,i64,list[struct[2]],list[struct[3]],list[struct[5]],list[struct[2]],list[struct[1]],f64,str,str,str
"""ENST00000309758.6""",352,17,37,335,232,4,15,228,"""HSPA6""","""HGNC:5239""","""protein_coding""","""ENSG00000173110.8""","[""basic"", ""Ensembl_canonical"", … ""CCDS""]","""+""",6,null,null,null,"""ensembl_havana_gene_homo_sapie…","[""MANE_select""]",1,"""FCHS_28105""","""chr1""",161523745,"""161524487""","""cp1@HSPA6""","[""GQ891451""]","""0""",53,"[{""EH38E2843853"",""pELS""}, {""EH38E2843855"",""PLS""}, {""EH38E3981533"",""pELS""}]","[{""STAG2"",4103,[{""SRX9975614"",4103}, {""SRX9975613"",3500}, {""SRX26159208"",562}]}, {""HSF1"",4038,[{""SRX16646095"",4038}, {""SRX16646111"",3572}, … {""SRX13845684"",554}]}, … {""GABPA"",500,[{""SRX2635792"",500}]}]","[{""HS_00013400"",""H1 BMP4 Derived Mesendoderm Cultured Cells"",""Mesendoderm cell"",""Others"",""SVM""}, {""HS_00013401"",""H1 Cells"",""Embryonic stem cell"",""Stem cell"",""SVM""}, … {""HS_00013403"",""Foreskin Fibroblast Primary Cells skin02"",""Prepuce"",""Skin"",""SVM""}]",null,null,0.982111,"""PLA""","""204,0,0""","""red"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ENST00000774581.1""",101,57,248,44,34,44,173,-10,null,null,"""lncRNA""","""ENSG00000300853.1""","[""basic"", ""TAGENE""]","""+""",2,"""URS0002A25289""","""ENSEMBL_GENCODE""",null,"""havana_tagene_homo_sapiens""","[""CLS3 project""]",1,"""FCHS_400104""","""chr8""",27592922,"""27593122""","""FCHS_400104""","[""ENST00000774579.1""]","""0""",294,"[{""EH38E3824983"",""dELS""}]",null,"[{""HS_00105821"",""NHEK-Epidermal Keratinocyte Primary Cells"",""Skin"",""Skin"",""SVM""}, {""HS_00105822"",""H1 BMP4 Derived Trophoblast Cultured Cells"",""Trophoblast cell"",""Others"",""SVM""}, … {""HS_00105828"",""Breast variant Human Mammary Epithelial Cells (vHMEC)"",""Breast"",""Breast"",""SVM""}]",null,null,0.5,"""ELA""","""255,202,74""","""yellow"""


In [6]:
up5p_fantabio.write_parquet(
    f"../../out/fantabio/parquet_file/up5p_transcript_fantabio_v{Execute_date}.parquet",
    compression_level=9
)

In [7]:
# check
HSPA1B = up5p_fantabio.filter(pl.col("gene_symbol") == "HSPA1B")
display(HSPA1B)


transcript_id,UP2,DOWN2,UNCHANGED2,HN2,UP2_stranded,DOWN2_stranded,UNCHANGED2_stranded,HN2_stranded,gene_symbol,HGNC_ID,transcript_type,gene_id,tags,strand,number_of_tags,RNACentral_ID,corrrespondence,polyA_features,gene_source,annotation_remarks,annotation_remark_count,cre_id,cre_chrom,cre_chrom_start,cre_chrom_end,cre_name,tss_nearest_transcript_ids,tss_nearest_transcript_distance,transcript_distance,screen_ccres,chip_atlas_antigens,silencerdbs,fantom5_cage_peaks,fantom5_enhancers,directionality,fantabio_cre_class,itemRgb,itemRgb_name
str,i64,i64,i64,i64,i64,i64,i64,i64,str,str,str,str,list[str],str,u32,str,str,list[str],str,list[str],u32,str,str,i64,str,str,list[str],str,i64,list[struct[2]],list[struct[3]],list[struct[5]],list[struct[2]],list[struct[1]],f64,str,str,str
"""ENST00000375650.5""",298,12,96,286,198,2,51,196,"""HSPA1B""","""HGNC:5233""","""protein_coding""","""ENSG00000204388.7""","[""basic"", ""Ensembl_canonical"", … ""CCDS""]","""+""",6,null,null,"[""polya_signal"", ""polya_site""]","""ensembl_havana_gene_homo_sapie…","[""MANE_select""]",1,"""FCHS_354485""","""chr6""",31826812,"""31827243""","""FCHS_354485""","[""ENST00000700791.2""]","""-200""",495,"[{""EH38E3700980"",""pELS""}]","[{""SOX10"",1998,[{""SRX13340348"",1998}, {""SRX13340342"",1533}, … {""SRX13340334"",1145}]}, {""STAT3"",1988,[{""SRX5323888"",1988}, {""SRX5323892"",1950}, … {""SRX5323838"",1456}]}, … {""CPSF3L"",505,[{""SRX9730737"",505}]}]",null,"[{""hg19::chr6:31794677..31794710,-;hg_72844.1"",""hg_72844.1""}, {""hg19::chr6:31795013..31795063,-;hg_72845.1"",""hg_72845.1""}]",null,-0.869084,"""PLA""","""0,0,204""","""blue"""
"""ENST00000375650.5""",298,12,96,286,198,2,51,196,"""HSPA1B""","""HGNC:5233""","""protein_coding""","""ENSG00000204388.7""","[""basic"", ""Ensembl_canonical"", … ""CCDS""]","""+""",6,null,null,"[""polya_signal"", ""polya_site""]","""ensembl_havana_gene_homo_sapie…","[""MANE_select""]",1,"""FCHS_354486""","""chr6""",31827058,"""31827936""","""cp1@HSPA1A,cp1@HSPA1B""","[""ENST00000375650.5"", ""NM_005346.6"", … ""BC009322""]","""0""",0,"[{""EH38E3700980"",""pELS""}, {""EH38E3700981"",""pELS""}, {""EH38E3700982"",""PLS""}]","[{""RFX1"",4363,[{""SRX2423398"",4363}, {""SRX2423399"",4127}, … {""SRX5457513"",1773}]}, {""HSF1"",4055,[{""SRX16646102"",4055}, {""SRX16646094"",3769}, … {""SRX8512780"",500}]}, … {""MYF5"",500,[{""SRX1962094"",500}]}]",null,"[{""hg19::chr6:31795013..31795063,-;hg_72845.1"",""hg_72845.1""}, {""hg19::chr6:31795123..31795151,-;hg_72846.1"",""hg_72846.1""}, … {""hg19::chr6:31795506..31795525,+;hg_67062.1"",""p2@HSPA1B""}]",null,0.997399,"""PLA""","""204,0,0""","""red"""


&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 4. No Proximal CRE Transcripts

- fanta.bio CRE annotation (± 500 bp) がない transcript を取得

In [8]:
# fanta.bio CRE annotation (± 500 bp) がない transcript を可視化
up5p_fantabio_no_cre = up5p_transcripts_list.select("transcript_id").unique().join(
    HNscore_transcript_all_fantabio,
    on="transcript_id",
    how="inner",
    coalesce=True
).sort(
    "HN2",
    descending=True
).filter(
    pl.col("cre_id").is_null() # Point: 近傍に fantabio CRE がない
)

# Drop columns that are null on every row (CRE fields stay null when cre_id is null)
up5p_fantabio_no_cre = up5p_fantabio_no_cre.select(
    [
        name
        for name, n_null in zip(
            up5p_fantabio_no_cre.columns,
            up5p_fantabio_no_cre.null_count().row(0),
        )
        if n_null < up5p_fantabio_no_cre.height
    ]
)

up5p_fantabio_no_cre.write_parquet(
    f"../../out/fantabio/no_proximal_cre_transcirpt_list/up5p_transcript_no_cre_v{Execute_date}.parquet",
    compression_level=9
)

display(up5p_fantabio_no_cre)

transcript_id,UP2,DOWN2,UNCHANGED2,HN2,UP2_stranded,DOWN2_stranded,UNCHANGED2_stranded,HN2_stranded,gene_symbol,HGNC_ID,transcript_type,gene_id,tags,strand,number_of_tags,RNACentral_ID,corrrespondence,polyA_features,gene_source,annotation_remarks,annotation_remark_count
str,i64,i64,i64,i64,i64,i64,i64,i64,str,str,str,str,list[str],str,u32,str,str,list[str],str,list[str],u32
"""ENST00000450744.1""",335,13,58,322,223,4,24,219,"""HSPA1B""","""HGNC:5233""","""protein_coding""","""ENSG00000232804.5""","[""basic""]","""+""",1,null,null,null,"""ensembl_havana_gene_homo_sapie…",null,0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ENST00000846985.1""",72,28,306,44,4,5,242,-1,"""PKP4-AS1""","""HGNC:52580""","""lncRNA""","""ENSG00000204380.5""","[""basic"", ""Ensembl_canonical"", … ""TAGENE""]","""-""",4,"""URS0002A3DDE3""","""ENSEMBL_GENCODE""",null,"""havana_homo_sapiens""","[""CLS3 project""]",1


&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 5. `ChIP-Atlas` Enrichment analysis で実行する用のBEDファイル作成

In [9]:
# https://data.fanta.bio/cre/v1.2.1/00README_v1.2.1.txt
fantabio_bed = pl.read_csv(
    "../../Data/Database/fantabio/human-CREv1.2.1.hg38.cre-peaks.bed",
    separator="\t",
    n_threads=16,
    has_header=False,
    new_columns=[
        "cre_chrom",
        "cre_chrom_start",
        "cre_chrom_end",
        "cre_id",
        "score",
        "strand",
        "thickStart",
        "thickEnd",
        "itemRgb",
        "cre_name",
        "pooled_cre_scores"
    ]
).with_columns(
    pl.col("pooled_cre_scores").str.extract(r"directionality:([^|]+)", 1).cast(pl.Float64, strict=False).alias("directionality"),
    pl.col("pooled_cre_scores").str.extract(r"class:([^|]+)", 1).alias("fantabio_cre_class"),
)

# Map exact RGB strings to human-readable names
rgb_to_name = {
    "153,0,0": "dark_red",
    "101,0,0": "dark_red",
    "50,0,0": "dark_red",
    "204,0,0": "red",
    "255,0,0": "red",
    "0,0,51": "dark_blue",
    "0,0,102": "dark_blue",
    "0,0,153": "blue",
    "0,0,204": "blue",
    "0,0,255": "blue",
    "0,0,0": "black",
    "255,202,74": "yellow",
}

fantabio_bed_select = fantabio_bed.with_columns(
    pl.col("itemRgb").replace_strict(
        rgb_to_name,
        default="other"
    ).alias("itemRgb_name")
).select(
    "cre_id",
    "cre_name",
    "directionality",
    "fantabio_cre_class",
    "itemRgb",
    "itemRgb_name"
)

display(fantabio_bed_select)

cre_id,cre_name,directionality,fantabio_cre_class,itemRgb,itemRgb_name
str,str,f64,str,str,str
"""FCHS_1""","""cp1@DDX11L1""",-1.0,"""PLA""","""0,0,255""","""blue"""
…,…,…,…,…,…
"""FCHS_593940""","""FCHS_593940""",0.989305,"""PLA""","""204,0,0""","""red"""


&nbsp;

&nbsp;

## 5-1: All CRE

- ChIP-Analysis Enrichment Analysis で使用するBEDファイルは、全ての transcript (up5p かつ、近傍に `fanta.bio` で定義されたCREが存在すること) のみを対象にすることにした

In [10]:
up5p_fantabio_bed = fantabio_bed.join(
    up5p_fantabio.select("cre_id").unique(),
    on="cre_id",
    how="semi"
)

print(f"up5p_fantabio_bed count: {up5p_fantabio_bed.select('cre_id').n_unique()}")
display(up5p_fantabio_bed)

up5p_fantabio_bed count: 33961


cre_chrom,cre_chrom_start,cre_chrom_end,cre_id,score,strand,thickStart,thickEnd,itemRgb,cre_name,pooled_cre_scores,directionality,fantabio_cre_class
str,i64,i64,str,i64,str,i64,i64,str,str,str,f64,str
"""chr1""",29398,29439,"""FCHS_2""",0,""".""",29398,29432,"""255,202,74""","""cp1@WASH7P""","""directionality:1|class:ELA""",1.0,"""ELA"""
…,…,…,…,…,…,…,…,…,…,…,…,…
"""chrY""",19076394,19076956,"""FCHS_447245""",0,""".""",19076666,19076785,"""255,202,74""","""cp5@TTTY14""","""directionality:0.777778|class:…",0.777778,"""ELA"""


&nbsp;

&nbsp;

## 5-2: Promoter Level Activity (PLA) CRE

In [11]:
up5p_fantabio_bed_pla = up5p_fantabio_bed.filter(
    pl.col("fantabio_cre_class") == "PLA"
)

up5p_fantabio_bed_pla.select(
    "cre_chrom",
    "cre_chrom_start",
    "cre_chrom_end"
).write_csv(
    f"../../out/fantabio/BEDfile/up5p/fantabio_up5p_all_pla_v{Execute_date}.bed",
    include_header=False,
    separator="\t"
)

print(f"up5p_fantabio_bed_pla count: {up5p_fantabio_bed_pla.select('cre_id').n_unique()}")
display(up5p_fantabio_bed_pla)

up5p_fantabio_bed_pla count: 14620


cre_chrom,cre_chrom_start,cre_chrom_end,cre_id,score,strand,thickStart,thickEnd,itemRgb,cre_name,pooled_cre_scores,directionality,fantabio_cre_class
str,i64,i64,str,i64,str,i64,i64,str,str,str,f64,str
"""chr1""",778536,778928,"""FCHS_19""",0,""".""",778631,778800,"""0,0,51""","""cp5@LINC00115,cp2@LINC01409""","""directionality:-0.304838|class…",-0.304838,"""PLA"""
…,…,…,…,…,…,…,…,…,…,…,…,…
"""chrY""",19075819,19076450,"""FCHS_447243""",0,""".""",19076001,19076187,"""0,0,204""","""cp4@TTTY14""","""directionality:-0.890534|class…",-0.890534,"""PLA"""


&nbsp;

&nbsp;

## 5-3: Enhancer Level Activity (ELA) CRE

In [12]:
up5p_fantabio_bed_ela = up5p_fantabio_bed.filter(
    pl.col("fantabio_cre_class") == "ELA"
)

up5p_fantabio_bed_ela.select(
    "cre_chrom",
    "cre_chrom_start",
    "cre_chrom_end"
).write_csv(
    f"../../out/fantabio/BEDfile/up5p/fantabio_up5p_all_ela_v{Execute_date}.bed",
    include_header=False,
    separator="\t"
)

print(f"up5p_fantabio_bed_ela count: {up5p_fantabio_bed_ela.select('cre_id').n_unique()}")
display(up5p_fantabio_bed_ela)

up5p_fantabio_bed_ela count: 19341


cre_chrom,cre_chrom_start,cre_chrom_end,cre_id,score,strand,thickStart,thickEnd,itemRgb,cre_name,pooled_cre_scores,directionality,fantabio_cre_class
str,i64,i64,str,i64,str,i64,i64,str,str,str,f64,str
"""chr1""",29398,29439,"""FCHS_2""",0,""".""",29398,29432,"""255,202,74""","""cp1@WASH7P""","""directionality:1|class:ELA""",1.0,"""ELA"""
…,…,…,…,…,…,…,…,…,…,…,…,…
"""chrY""",19076394,19076956,"""FCHS_447245""",0,""".""",19076666,19076785,"""255,202,74""","""cp5@TTTY14""","""directionality:0.777778|class:…",0.777778,"""ELA"""
